In [0]:
# MAGIC ## 1. Setup
from pyspark.sql import functions as F

SOURCE_PATH = "/databricks-datasets/retail-org/customers/"
TARGET_TABLE = "retail_dev.bronze.bronze_customers"


In [0]:
# MAGIC ## 2. Schema e Volume
df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("sep", ",")
    .csv(SOURCE_PATH)
)

print("=== SCHEMA ===")
df.printSchema()

print(f"\n=== VOLUME ===")
print(f"Total de linhas:   {df.count()}")
print(f"Total de colunas:  {len(df.columns)}")
print(f"Colunas:           {df.columns}")

In [0]:
# MAGIC ## 3. Amostra
display(df.limit(10))


In [0]:
# MAGIC ## 4. Qualidade — Nulos e Unicidade
# Nulos por coluna
display(
    df.select([
        F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
        for c in df.columns
    ])
)
# Clientes duplicados?
display(
    df.groupBy("customer_id")
    .agg(F.count("*").alias("ocorrencias"))
    .filter(F.col("ocorrencias") > 1)
    .orderBy(F.col("ocorrencias").desc())
)

In [0]:
# MAGIC ## 5. Análise
# Total de clientes únicos
print(f"Clientes únicos: {df.select('customer_id').distinct().count()}")

# Distribuição geográfica (colunas com state, city, country, region)
geo_cols = [c for c in df.columns if any(k in c.lower() for k in ["state", "city", "country", "region", "estado", "cidade"])]

if geo_cols:
    for col in geo_cols:
        print(f"\n=== {col} ===")
        display(
            df.groupBy(col)
            .agg(F.count("*").alias("total"))
            .orderBy(F.col("total").desc())
        )
else:
    print("Nenhuma coluna geográfica identificada. Colunas disponíveis:")
    for c in df.columns:
        print(f"  {c}")

In [0]:
# MAGIC ## 6. Ingestão → Delta
(
    df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TARGET_TABLE)
)

print(f"Tabela escrita: {TARGET_TABLE}")

In [0]:

# MAGIC ## 7. Verificação
df_check = spark.table(TARGET_TABLE)

print(f"Linhas na tabela: {df_check.count()}")
print(f"Colunas:          {df_check.columns}")

display(df_check.limit(5))
